In [2]:
import random
from matplotlib.pyplot import minorticks_on
from tqdm import tqdm
import src.generate_encodings as ge
import src.prediction_models as pm
import src.predictor_optimizer as pop
import src.mutant_discovery as dis
import numpy as np
import torch
import warnings
import os, sys
from sklearn.metrics import root_mean_squared_error

In [ ]:
n_layers = 35  #esmc_600 has 36 layers
embeddings  = []
data_set = "GRB2_HUMAN_Faure_2021"
mapping_file = f"../Data/Embeddings/{data_set}/pairings.csv"
method = "esm_650m"

In [5]:
class HiddenPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout


class HiddenWarnings():
    def __enter__(self):
        # Save the current filter settings before changing them
        self._previous_filters = warnings.filters[:]
        # Ignore all warnings
        warnings.filterwarnings("ignore")

    def __exit__(self, exc_type, exc_val, exc_tb):
        # Restore the original warning filter settings
        warnings.filters = self._previous_filters

In [17]:
import torch
from transformers import AutoModel, AutoTokenizer

model_path = 'Synthyra/FastESM2_650'
model = AutoModel.from_pretrained(model_path, torch_dtype=torch.float16, trust_remote_code=True).eval()
tokenizer = model.tokenizer

sequences = ['MPRTEIN', 'MSEQWENCE']
tokenized = tokenizer(sequences, padding=True, return_tensors='pt')
with torch.no_grad():
    embeddings = model(**tokenized, output_hidden_states=True).hidden_states[30]
                                                                        

    print(embeddings.shape) # (2, 11, 1280)

def mean_pooling(x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(x.size()).float()
    return torch.sum(x * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

seq_embeddings = mean_pooling(embeddings, tokenized['attention_mask'])
print(seq_embeddings.shape)  # (2, 1280)
# cls = embeddings[:, 0, :]

# print(cls.shape) # (2, 1280)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

modeling_fastesm.py:   0%|          | 0.00/51.4k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Synthyra/FastESM2_650:
- modeling_fastesm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/2.60G [00:00<?, ?B/s]

Some weights of FastEsmModel were not initialized from the model checkpoint at Synthyra/FastESM2_650 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.Size([2, 11, 1280])
torch.Size([2, 1280])


In [11]:
# for saved resultsfile
with open(f"../Results/esmc_600m_layer_comparison_{data_set_name_shortened}.csv", "r") as f:
    lines = f.readlines()[2:]
results = []
for line in lines:
    line = line[:-1].split(",")
    line[1] = int(float(line[1]))
    line[3] = float(line[3])
    line[4] = float(line[4])
    results.append(line)

models_tested = list(set([(result[0], result[2]) for result in results]))
# print(models_tested)
results = sorted(results, key=lambda x: x[1])
print(results)

[['xgboost', 1, ' tuned_hyperparams', 0.741, 0.215], ['lightgbm', 1, ' tuned_hyperparams', 0.755, 0.208], ['xgboost', 1, ' default_hyperparams', 0.73, 0.219], ['lightgbm', 1, ' default_hyperparams', 0.674, 0.239], ['rf', 1, ' default_hyperparams', 0.623, 0.264], ['adaboost', 1, ' default_hyperparams', 0.17, 0.385], ['xgboost', 2, ' tuned_hyperparams', 0.761, 0.207], ['lightgbm', 2, ' tuned_hyperparams', 0.783, 0.196], ['xgboost', 2, ' default_hyperparams', 0.759, 0.206], ['lightgbm', 2, ' default_hyperparams', 0.712, 0.228], ['rf', 2, ' default_hyperparams', 0.639, 0.253], ['adaboost', 2, ' default_hyperparams', 0.27, 0.363], ['xgboost', 3, ' tuned_hyperparams', 0.764, 0.206], ['lightgbm', 3, ' tuned_hyperparams', 0.786, 0.196], ['xgboost', 3, ' default_hyperparams', 0.756, 0.209], ['lightgbm', 3, ' default_hyperparams', 0.709, 0.231], ['rf', 3, ' default_hyperparams', 0.647, 0.251], ['adaboost', 3, ' default_hyperparams', 0.274, 0.359], ['xgboost', 4, ' tuned_hyperparams', 0.761, 0.20

In [12]:
"""Display Results: Modelperformance for every Layer in esmc_600m"""

from plotly.subplots import make_subplots
import plotly.graph_objects as go

results_plot = make_subplots(
    rows=1, cols=1)

colors = {
    ('xgboost', ' default_hyperparams'): "cyan",
    ('xgboost', ' tuned_hyperparams'): "darkcyan",
    ('rf', ' default_hyperparams'): "yellow",
    ('lightgbm', ' default_hyperparams'): "red",
    ('lightgbm', ' tuned_hyperparams'): "darkred",
    ('adaboost', ' default_hyperparams'): "grey"
}

for i, model in enumerate(models_tested):
    x= [result[1] for result in results if result[0] in model and result[2] in model]
    y= [result[3] for result in results if result[0] in model and result[2] in model]

    results_plot.append_trace(
        go.Scatter(name="_".join(model),
                   x=x,
                   y=y,
                   marker=dict(color=colors[model], size=3),
                   mode="lines"), row=1, col=1)

#
results_plot.update_layout(
    title_text=f"Performance of selected tested models for {data_set} Dataset for all sequence representation layers for {method}.",
    title_font=dict(color="black", size=20),
    showlegend=True,
    paper_bgcolor='rgb(233,233,233)',
    plot_bgcolor='rgb(233,233,233)',
    width=1400,
    height=1000,
    legend=dict(font=dict(color="black",
                          size=12)))
#
results_plot.update_yaxes(
    dict(
        title_text="R2 Performance",
        title_font=dict(color="black"),
        range=[0, 1],
        color='black',
        showgrid=True,
        gridcolor='grey',
        griddash="dot",
        dtick=0.1,
        gridwidth=1)
)

results_plot.update_xaxes(
    dict(
        title_text="cycles",
        title_font=dict(color="black"),
        range=[1, 35],
        color='black',
        showgrid=True,
        gridcolor='grey',
        dtick=1,
        griddash="dot",
        gridwidth=1)
)

import plotly.io as pio

pio.renderers.default = "browser"
with HiddenWarnings():
    results_plot.show()

import kaleido
results_plot.write_image(f"ESMC_600m_layer_comparison_{data_set_name_shortened}.jpg")

Gtk-Message: 15:18:03.705: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.
[255482, Main Thread] WARNING: GTK+ module /snap/firefox/6103/gnome-platform/usr/lib/gtk-2.0/modules/libcanberra-gtk-module.so cannot be loaded.
GTK+ 2.x symbols detected. Using GTK+ 2.x and GTK+ 3 in the same process is not supported.: 'glib warning', file /build/firefox/parts/firefox/build/toolkit/xre/nsSigHandlers.cpp:201

(firefox_firefox:255482): Gtk-WARNING **: 15:18:03.755: GTK+ module /snap/firefox/6103/gnome-platform/usr/lib/gtk-2.0/modules/libcanberra-gtk-module.so cannot be loaded.
GTK+ 2.x symbols detected. Using GTK+ 2.x and GTK+ 3 in the same process is not supported.
Gtk-Message: 15:18:03.755: Failed to load module "canberra-gtk-module"
[255482, Main Thread] WARNING: GTK+ module /snap/firefox/6103/gnome-platform/usr/lib/gtk-2.0/modules/libcanberra-gtk-module.so cannot be loaded.
GTK+ 2.x symbols detected. Using GTK+ 2.x and GTK+ 3 in the s